In [1]:
import torch
import torch.nn as nn
import timm
from timm.data.config import resolve_data_config
from timm.data.transforms_factory import create_transform
from transformers.modeling_outputs import ImageClassifierOutput

/home/chengxiaozhen/miniconda3/envs/HF/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class CLIP_ViT(nn.Module):
    def __init__(self,backbone_name="vit_small_patch16_384.augreg_in21k_ft_in1k", num_classes=1, fix_backbone=False):
        super(CLIP_ViT, self).__init__()
        self.vit = timm.create_model(backbone_name, pretrained=True)
        self.vit.head = nn.Linear(in_features=384, out_features=num_classes,bias=True)
    def forward(self, pixel_values, labels=None):
        logits = self.vit(pixel_values)
        return ImageClassifierOutput(
            loss=None,
            logits=logits,
        )

    def get_image_processor_or_transform(self):
        config = resolve_data_config({}, model=self.vit)

        train_transform = create_transform(**config, is_training=True)
        eval_transform = create_transform(**config, is_training=False)

        return train_transform, eval_transform


In [3]:
model = CLIP_ViT()

In [4]:
checkpoint_path = "/home/chengxiaozhen/Test/SFT-Infra/logs/CLIP/clip384/final_model/model.safetensors"
from safetensors.torch import load_file
state_dict=  load_file(str(checkpoint_path))
load_result = model.load_state_dict(state_dict, strict=False)
if load_result.missing_keys:
    print(f"Missing keys when loading checkpoint: {load_result.missing_keys}")
if load_result.unexpected_keys:
    print(f"Unexpected keys when loading checkpoint: {load_result.unexpected_keys}")


In [5]:
train_transform, eval_transform = model.get_image_processor_or_transform()
print(train_transform)

Compose(
    RandomResizedCropAndInterpolation(size=(384, 384), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bicubic)
    RandomHorizontalFlip(p=0.5)
    ColorJitter(brightness=(0.6, 1.4), contrast=(0.6, 1.4), saturation=(0.6, 1.4), hue=None)
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)


In [12]:
print(eval_transform)

Compose(
    Resize(size=384, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(384, 384))
    MaybeToTensor()
    Normalize(mean=tensor([0.5000, 0.5000, 0.5000]), std=tensor([0.5000, 0.5000, 0.5000]))
)
